In [261]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
import math


In [262]:

# 1. Load Data
df = pd.read_csv("./data/train_val.csv")


In [263]:
final_features = [
        'vol_rolling_21d', 'msft_return', 'dist_from_ma200', 'vol_garman_klass', 
        'qa_dispersion',
        'vix_level', 'fcf_margin', 'yield_10y_level', 'prepped_neutral',
        'days_since_earnings', 'cash_coverage', 'qqq_vol_21d', 'qa_sentiment',
        'qa_neutral', 'MDA_Delta'
    ]

# Basic Descriptive Stats about Features

In [264]:
# EDA on train_df
pd.set_option('display.max_columns',40)
print(" Dataset Overview:")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print("\nData Types:")
print(df.dtypes.value_counts())
print("\nFirst 5 rows:")
display(df.head().round(3))

print("\nMissing Values Summary:")
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print(missing)
else:
    print("No missing values found.")

print("\nSummary Statistics for Numerical Features:")
display(df.select_dtypes(include=[np.number]).describe().round(3))

 Dataset Overview:
Shape: (1180, 39)
Columns: 39

Data Types:
float64    37
str         2
Name: count, dtype: int64

First 5 rows:


,trading_date,msft_return,roa,cash_coverage,debt_to_asset,fcf_margin,net_income_qoq,doc_id,MDA_Neutrality,Risk_Combined_Neutrality,MDA_Sentiment,Risk_Combined_Mean,MDA_Dispersion,Risk_Combined_Std,MDA_Delta,Risk_Delta,prepped_sentiment,prepped_dispersion,prepped_neutral,qa_sentiment,qa_dispersion,qa_neutral,days_since_earnings,days_since_filing,qqq_return,spy_return,msft_vs_tech,msft_vs_market,vix_level,vix_5d_trend,yield_10y_level,yield_10y_delta_5d,vol_rolling_21d,qqq_vol_21d,vol_garman_klass,vol_surge,amihud_ratio,dist_from_ma200,target_vol_21d
0,2021-01-04,-0.021,0.046,0.109,0.59,0.388,0.24,NaN,0.225,0.304,-0.108,-0.58,0.719,0.378,-0.346,0.008,0.289,0.234,0.686,0.198,0.191,0.769,68.0,68.0,-0.014,-0.014,-0.007,-0.008,26.97,5.44,0.917,-0.011,0.180,0.122,0.166,1.351,0.0,0.091,0.283
1,2021-01-05,0.001,0.046,0.109,0.59,0.388,0.24,NaN,0.225,0.304,-0.108,-0.58,0.719,0.378,-0.346,0.008,0.289,0.234,0.686,0.198,0.191,0.769,69.0,69.0,0.008,0.007,-0.007,-0.006,25.34,3.64,0.955,0.022,0.179,0.125,0.166,0.869,0.0,0.090,0.284
2,2021-01-06,-0.026,0.046,0.109,0.59,0.388,0.24,NaN,0.225,0.304,-0.108,-0.58,0.719,0.378,-0.346,0.008,0.289,0.234,0.686,0.198,0.191,0.769,70.0,70.0,-0.014,0.006,-0.012,-0.032,25.07,1.99,1.042,0.107,0.202,0.135,0.172,1.285,0.0,0.060,0.261
3,2021-01-07,0.028,0.046,0.109,0.59,0.388,0.24,NaN,0.225,0.304,-0.108,-0.58,0.719,0.378,-0.346,0.008,0.289,0.234,0.686,0.198,0.191,0.769,71.0,71.0,0.024,0.015,0.004,0.014,22.37,-0.40,1.071,0.145,0.225,0.157,0.176,0.985,0.0,0.088,0.249
4,2021-01-08,0.006,0.046,0.109,0.59,0.388,0.24,NaN,0.225,0.304,-0.108,-0.58,0.719,0.378,-0.346,0.008,0.289,0.234,0.686,0.198,0.191,0.769,72.0,72.0,0.013,0.006,-0.007,0.000,21.56,-1.19,1.105,0.188,0.225,0.162,0.175,0.817,0.0,0.093,0.249



Missing Values Summary:
doc_id    1161
dtype: int64

Summary Statistics for Numerical Features:


,msft_return,roa,cash_coverage,debt_to_asset,fcf_margin,net_income_qoq,MDA_Neutrality,Risk_Combined_Neutrality,MDA_Sentiment,Risk_Combined_Mean,MDA_Dispersion,Risk_Combined_Std,MDA_Delta,Risk_Delta,prepped_sentiment,prepped_dispersion,prepped_neutral,qa_sentiment,qa_dispersion,qa_neutral,days_since_earnings,days_since_filing,qqq_return,spy_return,msft_vs_tech,msft_vs_market,vix_level,vix_5d_trend,yield_10y_level,yield_10y_delta_5d,vol_rolling_21d,qqq_vol_21d,vol_garman_klass,vol_surge,amihud_ratio,dist_from_ma200,target_vol_21d
count,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.000,1180.0,1180.000,1180.000
mean,0.001,0.048,0.121,0.506,0.297,0.041,0.236,0.332,0.253,-0.554,0.558,0.373,0.013,0.003,0.446,0.267,0.530,0.177,0.158,0.770,44.823,44.919,0.001,0.001,0.000,0.000,19.460,-0.031,3.323,0.013,0.248,0.212,0.188,1.003,0.0,0.075,0.247
std,0.016,0.004,0.028,0.045,0.102,0.087,0.088,0.021,0.234,0.020,0.131,0.016,0.234,0.018,0.062,0.028,0.062,0.056,0.060,0.055,26.779,26.787,0.014,0.011,0.009,0.011,5.393,3.593,1.172,0.137,0.087,0.092,0.051,0.338,0.0,0.116,0.087
min,-0.077,0.043,0.062,0.428,0.093,-0.109,0.102,0.290,-0.280,-0.594,0.317,0.344,-0.513,-0.035,0.289,0.195,0.433,0.102,0.069,0.623,0.000,0.000,-0.062,-0.059,-0.066,-0.070,11.860,-22.210,0.917,-0.484,0.100,0.073,0.113,0.305,0.0,-0.208,0.100
25%,-0.008,0.046,0.096,0.477,0.194,-0.019,0.175,0.316,0.134,-0.566,0.464,0.355,-0.153,-0.013,0.384,0.251,0.479,0.138,0.109,0.734,21.000,21.000,-0.007,-0.005,-0.005,-0.006,15.487,-1.600,2.002,-0.067,0.186,0.148,0.148,0.796,0.0,-0.011,0.183
50%,0.001,0.048,0.132,0.499,0.337,0.049,0.248,0.332,0.250,-0.556,0.553,0.376,0.000,0.002,0.468,0.276,0.509,0.164,0.152,0.769,44.000,44.000,0.001,0.001,-0.000,-0.000,18.330,-0.250,3.790,0.018,0.233,0.187,0.174,0.918,0.0,0.102,0.231
75%,0.010,0.050,0.140,0.544,0.360,0.110,0.269,0.341,0.411,-0.537,0.685,0.387,0.187,0.019,0.498,0.284,0.593,0.206,0.210,0.809,69.000,69.000,0.009,0.006,0.004,0.005,22.188,1.355,4.274,0.102,0.297,0.262,0.222,1.110,0.0,0.169,0.297
max,0.101,0.061,0.155,0.590,0.413,0.246,0.492,0.376,0.645,-0.517,0.749,0.394,0.519,0.037,0.541,0.312,0.686,0.335,0.297,0.852,97.000,97.000,0.120,0.105,0.066,0.077,52.330,30.560,4.988,0.511,0.552,0.597,0.348,2.999,0.0,0.326,0.552


In [ ]:


print("Quantitative EDA on Training Data...")

# Ensure dates are datetime objects and sort
df['trading_date'] = pd.to_datetime(df['trading_date'])
df = df.sort_values('trading_date')

# Define our core feature groups for targeted analysis``
target = 'target_vol_21d'
core_technicals = ['vol_rolling_21d', 'days_since_earnings', 'vol_garman_klass','dist_from_ma200','qa_neutral']
macro_liquidity = ['vix_level', 'yield_10y_delta_5d', 'qqq_vol_21d']

# Set aesthetic style
sns.set_theme(style="whitegrid", palette="muted")


print("\n📊 PILLAR 1: Target Variable Dynamics")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

#Distribution (Looking for skewness and fat tails)
sns.histplot(df[target], bins=50, kde=True, ax=axes[0], color='purple')
axes[0].set_title('Distribution of 21-Day Forward Volatility')
axes[0].set_xlabel('Annualized Volatility')

# 1B: Time Series (Looking for regime shifts and clustering)
axes[1].plot(df['trading_date'], df[target], color='purple', linewidth=1)
axes[1].set_title('Volatility Environment Over Time')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Annualized 21-Day ForwardVolatility')
plt.tight_layout()
plt.show()


#FEATURE-TARGET LINEARITY (SCATTER & HEXBIN)

print("\nFeature vs. Target Relationships")

# 1. Calculate the grid size dynamically based on how many features you have
n_features = len(final_features)
cols = 2
rows = math.ceil(n_features / cols)

# 2. Create the grid layout properly
fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
fig.suptitle("Correlation: Hexbin Density & Linear Trend", fontsize=16, fontweight='bold')
axes = axes.flatten() # Flatten so we can easily iterate over them

for i, feature in enumerate(final_features):
    ax = axes[i]
    
    # 3. Plot the Hexbin and the Regression line on the specific grid axis
    ax.hexbin(df[feature], df[target], gridsize=30, cmap='Blues', mincnt=1,vmin=-15)
    sns.regplot(data=df, x=feature, y=target, scatter=False, color='darkred', line_kws={'linewidth': 2}, ax=ax)
    
    ax.set_title(f'{feature} vs Target', fontweight='bold')
    ax.set_xlabel(feature)
    
    if i % cols == 0:
        ax.set_ylabel("Target 21D Volatility")
    else:
        ax.set_ylabel("")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.subplots_adjust(top=0.92) # Leave room for the main title
plt.show()

# MULTICOLLINEARITY & CLUSTERING
print("\n Feature Correlation Matrix (Spearman)")
# We use Spearman (Rank) correlation because financial relationships have non-linear relationships
features_for_corr = core_technicals + macro_liquidity + [target]
corr_matrix = df[features_for_corr].corr(method='spearman')

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', 
            vmax=1, vmin=-1, center=0, square=True, linewidths=.5)
plt.title("Spearman Rank Correlation (Non-Linear Inter-Feature Relationships)")
plt.show()





In [ ]:

def generate_model_driven_eda(df, final_features,target_col='target_vol_21d', date_col='trading_date'):
    """
    Generates presentation-ready EDA charts focusing strictly on the final pruned features.
    """
    print("Generating Model-Driven EDA for Final Presentation...")
    
    
    
    # Ensure date is sorted
    df_eda = df.copy()
    df_eda[date_col] = pd.to_datetime(df_eda[date_col])
    df_eda = df_eda.sort_values(date_col).set_index(date_col)

    # ==========================================
    # CHART 1: Alpha Correlation Profile
    # ==========================================
    plt.figure(figsize=(12, 8))
    
    # Calculate correlation to target, drop the target itself, and sort
    correlations = df_eda[final_features + [target_col]].corr(method='spearman')[target_col].drop(target_col)
    correlations = correlations.sort_values()
    
    # Color code: Blue for positive correlation (increases vol), Red for negative (decreases vol)
    colors = ['red' if x < 0 else 'blue' for x in correlations]
    
    correlations.plot(kind='barh', color=colors, width=0.7)
    plt.axvline(0, color='black', linewidth=1)
    plt.title('Linear Alpha Profile: Feature Correlation to 21-Day Volatility', pad=20, fontsize=14, fontweight='bold')
    plt.xlabel('Spearman Correlation Coefficient')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ==========================================
    # CHART 2: Time-Series Regime Overlay (The Narrative Chart)
    # ==========================================
    # We plot the Target Volatility against your top Macro (VIX) and top NLP (qa_dispersion)
    print("Generating Split-Panel Regime Chart...")
    
    # Create a figure with 2 subplots (Top is larger, Bottom is smaller)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]}, sharex=True)
    
    # --- TOP PANEL: The Macro Regime (Daily Frequency) ---
    # Plot Target Volatility (Background Fill)
    ax1.fill_between(df_eda.index, df_eda[target_col], color='grey', alpha=0.5, label='Actual 21D Volatility')
    ax1.plot(df_eda.index, df_eda[target_col], color='grey', linewidth=1)
    
    # Plot VIX on a secondary axis for the top panel
    ax1_vix = ax1.twinx()
    ax1_vix.plot(df_eda.index, df_eda['vix_level'], color='blue', linewidth=1.5, alpha=0.8, label='VIX Level')
    
    ax1.set_ylabel('Annualized Volatility', fontweight='bold', fontsize=12)
    ax1_vix.set_ylabel('VIX Level', fontweight='bold', fontsize=12, color='blue')
    ax1.set_title('Macro Regime & NLP Alpha Alignment', pad=15, fontsize=16, fontweight='bold')
    
    # Combine Top Panel Legends
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_vix, labels_vix = ax1_vix.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_vix, labels_1 + labels_vix, loc='upper left')
    
    # --- BOTTOM PANEL: The NLP Alpha Signal (Quarterly Step Frequency) ---
    # Plot Q&A dispersion as a filled step graph to correctly represent forward-filled states
    ax2.fill_between(df_eda.index, df_eda['qa_dispersion'], step='post', color='#d9534f', alpha=0.3)
    ax2.step(df_eda.index, df_eda['qa_dispersion'], where='post', color='#d9534f', linewidth=2, label='Q&A Dispersion (Raw Signal)')
    
    # Add a zero-line to easily see positive/negative dispersion regimes
    ax2.axhline(df_eda['qa_dispersion'].mean(), color='black', linestyle='--', linewidth=1, alpha=0.5, label='Signal Mean')
    
    ax2.set_ylabel('NLP Dispersion Score', fontweight='bold', fontsize=12)
    ax2.legend(loc='upper left')
    
    # Clean up the grid
    ax1.grid(alpha=0.3)
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    # ==========================================
    # CHART 3: Alpha Scatter Grid (Proving the specific relationships)
    # ==========================================
    # Select the Top 3 unique non-price drivers to show nonlinear relationships
    top_scatter_features = ['qa_dispersion', 'vix_level', 'yield_10y_level']
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Bivariate Signal Analysis: Top Alpha Generators vs Target Volatility', fontsize=14, fontweight='bold', y=1.05)
    
    for i, feature in enumerate(top_scatter_features):
        sns.regplot(
            data=df_eda, 
            x=feature, 
            y=target_col, 
            ax=axes[i], 
            scatter_kws={'alpha':0.3, 'color':'#2e5597'}, 
            line_kws={'color':'#d9534f', 'linewidth':2}
        )
        axes[i].set_title(f'{feature}', fontweight='bold')
        axes[i].set_ylabel('Target 21D Volatility' if i == 0 else '')
        axes[i].grid(alpha=0.3)
        
    plt.tight_layout()
    plt.show()

# Execution
# generate_model_driven_eda(df_merged)

In [ ]:
generate_model_driven_eda(df)

In [ ]:
df.columns

In [ ]:

# Set the Institutional Background Theme
sns.set_theme(style="whitegrid")
plt.rcParams['figure.facecolor'] = 'white' # Clean white background for slides
plt.rcParams['axes.facecolor'] = 'white'

# ==========================================
# ANALYSIS 1: CONTINUOUS FEATURES (Price & Macro)
# Tool: Hexbin Scatter Plots with Regression Lines
# ==========================================
print("\nAnalyzing Continuous Daily Features (Price & Macro)...")

# FIXED: Removed step-functions from this list so hexbins look organic
top_continuous = [
    'vol_garman_klass', 'msft_return', 'vix_level', 
    'dist_from_ma200', 'yield_10y_level', 'vol_rolling_21d'
]

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle("Leverage & Shocks: Continuous Feature Density", fontsize=16, fontweight='bold')
axes = axes.flatten()

for i, feature in enumerate(top_continuous):
    ax = axes[i]
    # Use 'Blues' colormap for density, 'darkred' for the trendline
    ax.hexbin(df[feature], df[target], gridsize=25, cmap='Blues', mincnt=1,vmin=-15)
    sns.regplot(data=df, x=feature, y=target, scatter=False, color='darkred', line_kws={'linewidth': 2}, ax=ax)
    
    ax.set_title(f"{feature} vs Future Volatility", fontweight='bold')
    ax.set_ylabel("Target 21D Volatility")

plt.tight_layout()
plt.subplots_adjust(top=0.92) # Leave room for the main title
plt.show()

# ==========================================
# ANALYSIS 2: STEP-FUNCTION FEATURES (NLP & Fundamentals)
# Tool: Quintile Spread Boxplots
# ==========================================
print("\nAnalyzing Step-Function Features (NLP & Fundamentals)...")

top_step = ['prepped_neutral', 'qa_dispersion', 'qa_neutral', 'MDA_Delta']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Risk Spread by Quintile: NLP & Document Deltas", fontsize=16, fontweight='bold')
axes = axes.flatten()

for i, feature in enumerate(top_step):
    ax = axes[i]
    
    # Create 5 buckets (Quintiles) using rank to handle forward-filled identical values
    df['temp_quintile'] = pd.cut(df[feature], bins=5, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4', 'Q5 (High)'])
    
    # Use 'Blues' palette for a clean, corporate gradient
    sns.boxplot(data=df, x='temp_quintile', y=target, palette='Blues', ax=ax)
    
    ax.set_title(f"Volatility Distribution: {feature}", fontweight='bold')
    ax.set_xlabel(f"{feature} Quintiles")
    ax.set_ylabel("Target 21D Volatility")

df.drop(columns=['temp_quintile'], inplace=True)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

# ==========================================
# ANALYSIS 3: THE TEMPORAL CLOCK
# Tool: Dual-Axis Time Series
# ==========================================
print("\nAnalyzing Information Decay (Temporal Clock)...")

# Ensure trading date is a datetime object for clean x-axis formatting
df['trading_date'] = pd.to_datetime(df['trading_date'])

# Take a 1-year slice (roughly 252 trading days)
slice_df = df[df.trading_date.dt.year==2024].copy()

fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot Days Since Earnings on primary axis (Bottom) using a solid named color
ax1.plot(slice_df['trading_date'], slice_df['days_since_earnings'], color='steelblue', linewidth=2, label='Days Since Earnings')
ax1.set_ylabel('Days Since Last Earnings Call', color='steelblue', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='steelblue')

# Plot Future Volatility on secondary axis (Top)
ax2 = ax1.twinx()
ax2.plot(slice_df['trading_date'], slice_df[target], color='darkred', linewidth=1.5, alpha=0.8, label='Future 21D Volatility')
ax2.set_ylabel('Future 21-Day Volatility', color='darkred', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='darkred')

# Combine legends properly for a dual-axis chart
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left', frameon=True, facecolor='white')

plt.title("The 'Information Decay' Cycle: Volatility Expansion Post-Earnings", pad=15, fontsize=14, fontweight='bold')
plt.grid(False) # Turn off the secondary grid to avoid cross-hatching
plt.tight_layout()
plt.show()

In [ ]:


def investigate_fat_tails(df, feature='qa_dispersion', target='target_vol_21d', tail_percentile=0.90):
    """
    Isolates the most chaotic regime (Q5) and extracts the exact dates 
    where the fat-tail volatility blowups actually occurred.
    """
    print(f"🕵️ Initiating Fat Tail Autopsy for: {feature}\n")
    
    # 1. Recreate the Quintiles (using pd.cut as we discussed earlier)
    df_temp = df.copy()
    df_temp['temp_quintile'] = pd.cut(df_temp[feature], bins=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
    
    # 2. Isolate ONLY the highest risk regime (Q5)
    q5_data = df_temp[df_temp['temp_quintile'] == 'Q2']
    
    # 3. Define the "Fat Tail" (e.g., the top 10% worst days within Q5)
    tail_threshold = q5_data[target].quantile(tail_percentile)
    
    # 4. Filter for the exact days that broke the threshold
    fat_tail_events = q5_data[q5_data[target] >= tail_threshold].copy()
    
    # 5. Extract just the Year and Month for a clean report
    fat_tail_events['Year-Month'] = pd.to_datetime(fat_tail_events['trading_date']).dt.to_period('M')
    
    print(f"--- Q5 REGIME: EXTREME EVENT DATES ---")
    print(f"Threshold for 'Fat Tail' set at Volatility > {tail_threshold:.4f}\n")
    
    # Group by month to see when the clusters happened
    event_clusters = fat_tail_events.groupby('Year-Month').size().reset_index(name='Days in Fat Tail')
    print(event_clusters.sort_values('Year-Month', ascending=False).to_string(index=False))
    
    # Optional: Plot them on a timeline to visualize the clusters
    plt.figure(figsize=(12, 4))
    plt.plot(pd.to_datetime(df_temp['trading_date']), df_temp[target], color='lightgrey', label='All Volatility')
    plt.scatter(pd.to_datetime(fat_tail_events['trading_date']), fat_tail_events[target], color='darkred', label='Q5 Fat Tails', zorder=5)
    plt.title(f"Chronological Map of Q5 Fat Tail Events ({feature})", fontweight='bold')
    plt.ylabel("Target 21D Volatility")
    plt.legend()
    plt.tight_layout()
    plt.show()

# Run it on your dataframe
investigate_fat_tails(df, feature='qa_dispersion')

In [ ]:

from scipy.interpolate import griddata


print("Generating Macro vs Micro Regime Matrix...")


# 1. Create 33rd and 66th percentile bins (Low, Medium, High)
df['macro_vix_regime'] = pd.qcut(df['dist_from_ma200'], q=3, labels=['1. Low Rates', '2. Med Rates', '3. High Rates'])
# df['macro_vix_regime'] = pd.qcut(df['vix_level'], q=3, labels=['1. Low VIX', '2. Med VIX', '3. High VIX'])
df['micro_sent_regime'] = pd.cut(df['qa_dispersion'], bins=3, labels=['Low', '2. Neutral Tone', '3. High Tone'])

# 2. Build the Pivot Table (Calculate average target volatility for each intersection)
regime_matrix = pd.pivot_table(
    df, 
    values='target_vol_21d', 
    index='macro_vix_regime', 
    columns='micro_sent_regime', 
    aggfunc=np.mean
)

# 3. Plot the Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(regime_matrix, annot=True, fmt=".3f", cmap="YlOrRd", cbar_kws={'label': 'Average Future 21-Day Volatility'})
plt.title("The 2D Risk Matrix: Macro Fear vs. Executive Sentiment")
plt.xlabel("Microsoft Executive Tone (NLP)")
plt.ylabel("Global Macro Regime (VIX)")
plt.gca().invert_yaxis() # Put High VIX at the top for intuitive reading
plt.tight_layout()
plt.show()

# Cleanup
df.drop(columns=['macro_vix_regime', 'micro_sent_regime'], inplace=True)

In [ ]:
df = df[df['trading_date'].dt.year>=2021]

In [ ]:
df.columns

In [ ]:

print("Generating the Information Decay Curve...")

# 1. Create the NLP Regimes (using absolute cuts for step-functions)
# We use 3 bins here (Tertiles) instead of 5 to keep the chart clean and readable
df['nlp_regime'] = pd.cut(
    df['qa_sentiment'], 
    bins=3, 
    labels=['1. Low Dispersion (Clear)', '2. Neutral', '3. High Dispersion (Confused)']
)

# 2. Convert 'Days' to 'Weeks' to smooth out the daily noise
df['weeks_since_earnings'] = df['days_since_earnings'] // 7

# 3. Plot the Decay Curve
plt.figure(figsize=(12, 7))

# sns.lineplot automatically calculates the mean volatility for each week 
# and draws a shaded 95% confidence interval around the lines.
sns.lineplot(
    data=df, 
    x='weeks_since_earnings', 
    y=target, 
    hue='nlp_regime', 
    palette=['steelblue', 'gray', 'darkred'], 
    linewidth=2.5,
    estimator=np.median, 
    errorbar=None,
    marker='o',
    markersize=8
)

plt.title("Information Decay: NLP Executive Tone vs. Time Since Earnings", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Weeks Since Last Earnings Call")
plt.ylabel("Average Future 21-Day Volatility")

# Clean up the legend
plt.legend(title="Q&A Dispersion Regime", title_fontsize='12', fontsize='11', loc='upper right', frameon=True, facecolor='white')

# Set x-axis ticks to show the 12-week quarter lifecycle clearly
plt.xticks(range(0, 14))
plt.xlim(0, 13)

plt.tight_layout()
plt.show()

# Cleanup
df.drop(columns=['nlp_regime', 'weeks_since_earnings'], inplace=True)

In [ ]:

# 1. How does Sentiment affect Volatility? (Scatter Plot)
plt.figure(figsize=(8, 5))
sns.regplot(
    data=df, 
    x='cash_coverage', 
    y='target_vol_21d', 
    scatter_kws={'alpha':0.3}, 
    line_kws={'color':'red'}
)
plt.title("Training Set: Cash Coverage vs. Future Volatility")
plt.xlabel("Cash Coverage")
plt.ylabel("Future 21-Day Volatility")
plt.show()

# 2. How does 'Days Since Earnings' act over time? (Time Series)
# We plot a small slice (e.g., first 300 days) so we can see the "sawtooth" pattern
slice_df = df.tail(300)

fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()

ax1.plot(slice_df['trading_date'], slice_df['days_since_earnings'], color='blue', label='Days Since Earnings')
ax2.plot(slice_df['trading_date'], slice_df['target_vol_21d'], color='red', alpha=0.5, label='Volatility')

ax1.set_ylabel('Days Since Earnings', color='blue')
ax2.set_ylabel('Volatility', color='red')
plt.title("The 'Information Decay' Cycle")
plt.show()

In [ ]:
# QUINTILE ANALYSIS 
print("\n Quintile Spread Analysis")
# does the average risk step up linearly?

df['gk_quintile'] = pd.qcut(df['vol_garman_klass'], q=5, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4', 'Q5 (High)'], duplicates='drop')

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='gk_quintile', y=target, palette='viridis')
plt.title('Future Volatility Distribution by Garman-Klass Quintiles')
plt.xlabel('Garman-Klass Volatility Quintiles (Today)')
plt.ylabel('Future 21-Day Volatility')
plt.show()

# Cleanup temporary column
df.drop(columns=['gk_quintile'], inplace=True)
print("EDA Complete.")